# 02 · Atenção (uma cabeça)

**Peça nova:** self-attention. O bigrama ignora tudo antes do último caractere. Agora cada posição vai **juntar informação das posições anteriores**, com pesos que dependem do conteúdo.

Duas outras peças entram junto, porque a atenção precisa delas:
- **embedding de token** (`tok_emb`): cada caractere vira um vetor de `d_model` números;
- **embedding de posição** (`pos_emb`): um vetor aprendido para cada posição 0, 1, 2…, somado ao do token. Sem ele, a atenção não sabe a ordem.

Diário: `F1-ATN`.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from lab.data import shakespeare
from lab.model import LanguageModel, count_params
from lab.plot import compare, plot_attention, table
from lab.train import generate_text, pick_device, train

DEVICE = pick_device()
ds = shakespeare()
tok, V = ds.tok, ds.tok.vocab_size
print(ds, "| dispositivo:", DEVICE)

## O truque matemático: média do passado com uma multiplicação

A forma mais simples de usar o contexto é cada posição `t` tirar a **média** dos vetores das posições `0..t`. Abaixo, três maneiras de fazer a mesma conta. A terceira é a que a atenção usa: um softmax sobre "afinidades" com `-inf` nas posições futuras.

In [ ]:
torch.manual_seed(42)
B, T, C = 1, 8, 2
x = torch.randn(B, T, C)

# 1. laço explícito
xbow = torch.zeros(B, T, C)
for t in range(T):
    xbow[:, t] = x[:, : t + 1].mean(dim=1)

# 2. matriz triangular inferior normalizada por linha
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x

# 3. softmax de afinidades (todas zero) com -inf no futuro
aff = torch.zeros(T, T).masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))
wei3 = F.softmax(aff, dim=-1)
xbow3 = wei3 @ x

print(wei3.round(decimals=2))
print("as três iguais:", torch.allclose(xbow, xbow2), torch.allclose(xbow, xbow3))

Na versão 3 as afinidades são todas zero, então o peso é igual para todo o passado. A self-attention troca esses zeros por **afinidades calculadas a partir dos dados**:

- cada posição emite uma **query** ("o que estou procurando") e uma **key** ("o que eu tenho");
- a afinidade de `i` com `j` é `q_i · k_j`, dividida por `sqrt(head_size)`;
- o que se soma não é o vetor bruto `x_j`, e sim um **value** `v_j` ("o que eu entrego se você prestar atenção em mim").

## A peça: `Head`

In [ ]:
class Head(nn.Module):
    def __init__(self, d_model, head_size, block_size, causal=True):
        super().__init__()
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        self.causal = causal
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.last_att = None

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)        # (B, T, hs)
        att = q @ k.transpose(-2, -1) * k.size(-1) ** -0.5          # (B, T, T) afinidades
        if self.causal:
            att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # proibido olhar o futuro
        att = F.softmax(att, dim=-1)                                # cada linha soma 1
        self.last_att = att.detach()
        return att @ v                                              # (B, T, hs) média ponderada dos values

### Testes
1. Mudar os tokens **futuros** não pode mudar a saída das posições passadas.
2. Cada linha da matriz de atenção soma 1, e acima da diagonal é tudo zero.

In [ ]:
torch.manual_seed(0)
h = Head(d_model=16, head_size=8, block_size=8)
x = torch.randn(1, 8, 16)
y1 = h(x)
x2 = x.clone()
x2[:, 5:] = torch.randn(1, 3, 16)  # muda só as posições 5, 6 e 7
y2 = h(x2)
print("posições 0-4 iguais:", torch.allclose(y1[:, :5], y2[:, :5]))
print("posições 5-7 mudaram:", not torch.allclose(y1[:, 5:], y2[:, 5:]))
print(h.last_att[0].round(decimals=2))

### Por que dividir por `sqrt(head_size)`?
Se q e k têm componentes com variância 1, o produto escalar `q·k` tem variância `head_size`. Com afinidades grandes o softmax vira quase um "one-hot": cada posição olha para uma só e o gradiente das outras some.

In [ ]:
for hs in [4, 64, 512]:
    q, k = torch.randn(10_000, hs), torch.randn(10_000, hs)
    s = (q * k).sum(-1)
    print(f"head_size {hs:3d}: var(q·k) = {s.var():7.1f}   com a escala: {(s * hs ** -0.5).var():.2f}")
aff = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print("softmax(aff)     ", F.softmax(aff, -1).round(decimals=3))
print("softmax(aff * 8) ", F.softmax(aff * 8, -1).round(decimals=3))

## O modelo: embeddings + uma cabeça + saída

`LanguageModel` (de `lab.model`) fornece o `forward` com a loss e o `generate` do notebook 01. A única novidade do `generate` é cortar o contexto nos últimos `block_size` tokens, porque `pos_emb` só tem `block_size` posições.

In [ ]:
class AttentionLM(LanguageModel):
    def __init__(self, vocab_size, block_size=32, d_model=64, head_size=64, pos_emb=True, causal=True):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model) if pos_emb else None
        self.head = Head(d_model, head_size, block_size, causal=causal)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def logits(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)                                          # (B, T, d_model): o que é cada token
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))   # + onde ele está
        x = self.head(x)                                               # (B, T, head_size): mistura do passado
        return self.lm_head(x)                                         # (B, T, vocab)

## Treino de referência
Configuração usada daqui até o notebook 06: contexto de 32 caracteres, `d_model = 64`, 3000 passos, lr 1e-3.

In [ ]:
BLOCK, D, STEPS, LR = 32, 64, 3000, 1e-3

torch.manual_seed(1337)
model = AttentionLM(V, BLOCK, D, head_size=D)
run = train(model, ds, name="02-atencao", steps=STEPS, lr=LR, device=DEVICE)
print(run.sample)

In [ ]:
compare(["01-bigram", "02-atencao"], title="Bigrama vs. uma cabeça de atenção")
table(["01-bigram", "02-atencao"])

### Para onde a cabeça olha?
Cada linha é uma posição fazendo a pergunta; as colunas mostram quanto peso ela dá a cada caractere anterior.

In [ ]:
text = "First Citizen:\nBefore we proceed any further"[:BLOCK]
idx = torch.tensor([tok.encode(text)], device=DEVICE)
model.eval()
model(idx)
plot_attention(model.head.last_att[0].cpu(), list(text), title="02-atencao")
plt.show()

---
## Experimentos

### 1. Tamanho do contexto
**Minha previsão:** _(com 8, 32 e 128 caracteres de contexto, qual loss de validação? ganho diminui?)_

In [ ]:
for bs in [8, 32, 128]:
    torch.manual_seed(1337)
    train(AttentionLM(V, bs, D, D), ds, name=f"02-ctx{bs}", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
compare(prefix="02-ctx", title="Uma cabeça: block_size")
table(prefix="02-ctx")

### 2. Sem embedding de posição
Sem `pos_emb`, a atenção vê um **conjunto** de caracteres, não uma sequência.

**Minha previsão:** _(fica pior que o bigrama, igual ou entre o bigrama e o modelo com posição?)_

In [ ]:
torch.manual_seed(1337)
r = train(AttentionLM(V, BLOCK, D, D, pos_emb=False), ds, name="02-sem-posicao", steps=STEPS, lr=LR, device=DEVICE, verbose=False)
compare(["01-bigram", "02-atencao", "02-sem-posicao"])
table(["01-bigram", "02-atencao", "02-sem-posicao"])
print(r.sample)

### 3. Sem a máscara causal
Sem a máscara, a posição `t` enxerga o caractere `t+1`, que é exatamente o alvo dela.

**Minha previsão:** _(o que acontece com a loss de treino? e de validação? e com o texto gerado?)_

In [ ]:
torch.manual_seed(1337)
r = train(AttentionLM(V, BLOCK, D, D, causal=False), ds, name="02-sem-mascara", steps=STEPS, lr=LR, device=DEVICE, verbose=False)
compare(["02-atencao", "02-sem-mascara"])
table(["02-atencao", "02-sem-mascara"])
print(r.sample)

A loss de validação também cai porque ela é medida do mesmo jeito, com o futuro visível. Na geração não existe futuro: a última posição nunca tem o próximo caractere para copiar, e o modelo nunca aprendeu a prever sem ele. Visualize a atenção desse modelo para ver a "cola":

In [ ]:
cheat = r.model.eval()
cheat(idx)
plot_attention(cheat.head.last_att[0].cpu(), list(text), title="sem máscara")
plt.show()

### 4. Tamanho da cabeça (`head_size`)
**Minha previsão:** _(16, 64 ou 128 dimensões para q, k e v: quanto muda a loss? e o número de parâmetros?)_

In [ ]:
for hs in [16, 64, 128]:
    torch.manual_seed(1337)
    m = AttentionLM(V, BLOCK, D, head_size=hs)
    print(f"head_size {hs}: {count_params(m):,} parâmetros")
    train(m, ds, name=f"02-hs{hs}", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
compare(prefix="02-hs", title="Uma cabeça: head_size")
table(prefix="02-hs")

---
## Para o diário (`F1-ATN-01`)

- Quanto a atenção ganhou sobre o bigrama? Quanto desse ganho some sem a posição?
- Explique, com suas palavras, por que o modelo sem máscara "cola" e por que isso não ajuda a gerar texto.
- O que o mapa de atenção mostrou? A cabeça olha mais para os caracteres próximos?

**Próximo:** [03 · várias cabeças](03_varias_cabecas.ipynb).